# Préparation des fichiers CSV pour Neo4j
Ce notebook prépare les datasets de maladies, symptômes et précautions pour l'importation dans Neo4j.  
Il normalise les données, crée des relations et génère des fichiers CSV pour les nœuds et les relations du graphe.


1️⃣ Charger les datasets (cellule code)

In [5]:
import pandas as pd

# Charger le dataset Disease and Symptoms
df_symptoms = pd.read_csv("DiseaseAndSymptoms.csv")  # adapte le nom du fichier
print(df_symptoms.head())
print(df_symptoms.info())


            Disease   Symptom_1              Symptom_2              Symptom_3  \
0  Fungal infection     itching              skin_rash   nodal_skin_eruptions   
1  Fungal infection   skin_rash   nodal_skin_eruptions    dischromic _patches   
2  Fungal infection     itching   nodal_skin_eruptions    dischromic _patches   
3  Fungal infection     itching              skin_rash    dischromic _patches   
4  Fungal infection     itching              skin_rash   nodal_skin_eruptions   

              Symptom_4 Symptom_5 Symptom_6 Symptom_7 Symptom_8 Symptom_9  \
0   dischromic _patches       NaN       NaN       NaN       NaN       NaN   
1                   NaN       NaN       NaN       NaN       NaN       NaN   
2                   NaN       NaN       NaN       NaN       NaN       NaN   
3                   NaN       NaN       NaN       NaN       NaN       NaN   
4                   NaN       NaN       NaN       NaN       NaN       NaN   

  Symptom_10 Symptom_11 Symptom_12 Symptom_13 Symp

In [7]:
# Charger le dataset Disease Precaution
df_precaution = pd.read_csv("Disease precaution.csv")  # adapte le nom du fichier
print(df_precaution.head())
print(df_precaution.info())


          Disease                      Precaution_1  \
0   Drug Reaction                   stop irritation   
1         Malaria          Consult nearest hospital   
2         Allergy                    apply calamine   
3  Hypothyroidism                     reduce stress   
4       Psoriasis  wash hands with warm soapy water   

                   Precaution_2        Precaution_3  \
0      consult nearest hospital    stop taking drug   
1               avoid oily food  avoid non veg food   
2       cover area with bandage                 NaN   
3                      exercise         eat healthy   
4  stop bleeding using pressure      consult doctor   

                  Precaution_4  
0                    follow up  
1           keep mosquitos out  
2  use ice to compress itching  
3             get proper sleep  
4                   salt baths  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dty

2️⃣ Normalisation des données (cellule code)

In [9]:
import re

def normalize(s):
    if pd.isna(s):
        return None
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = s.strip(".,;")
    return s

# Normaliser maladies et symptômes
df_symptoms['Disease'] = df_symptoms['Disease'].map(normalize)
for i in range(1, 18):
    df_symptoms[f'Symptom_{i}'] = df_symptoms[f'Symptom_{i}'].map(normalize)

# Normaliser maladies et précautions
df_precaution['Disease'] = df_precaution['Disease'].map(normalize)
for i in range(1, 5):
    df_precaution[f'Precaution_{i}'] = df_precaution[f'Precaution_{i}'].map(normalize)


In [11]:
df_precaution

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,drug reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,malaria,consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,allergy,apply calamine,cover area with bandage,None,use ice to compress itching
3,hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths
5,gerd,avoid fatty spicy food,avoid lying down after eating,maintain healthy weight,exercise
6,chronic cholestasis,cold baths,anti itch medicine,consult doctor,eat healthy
7,hepatitis a,consult nearest hospital,wash hands through,avoid fatty spicy food,medication
8,osteoarthristis,acetaminophen,consult nearest hospital,follow up,salt baths
9,(vertigo) paroymsal positional vertigo,lie down,avoid sudden change in body,avoid abrupt head movment,relax


In [13]:
df_symptoms

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,None,None,None,None,None,None,None,None,None,None,None,None,None
1,fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,fungal infection,itching,nodal_skin_eruptions,dischromic _patches,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,fungal infection,itching,skin_rash,dischromic _patches,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,fungal infection,itching,skin_rash,nodal_skin_eruptions,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,(vertigo) paroymsal positional vertigo,vomiting,headache,nausea,spinning_movements,loss_of_balance,unsteadiness,None,None,None,None,None,None,None,None,None,None,None
4916,acne,skin_rash,pus_filled_pimples,blackheads,scurring,None,None,None,None,None,None,None,None,None,None,None,None,None
4917,urinary tract infection,burning_micturition,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,None,None,None,None,None,None,None,None,None,None,None,None,None
4918,psoriasis,skin_rash,joint_pain,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,None,None,None,None,None,None,None,None,None,None,None


3️⃣ Création des relations maladie → symptômes (cellule code)

In [15]:
rows = []
for _, row in df_symptoms.iterrows():
    disease = row['Disease']
    for i in range(1, 18):
        symptom = row[f'Symptom_{i}']
        if symptom:
            rows.append((disease, symptom))

df_disease_symptom = pd.DataFrame(rows, columns=['disease', 'symptom'])
df_disease_symptom.drop_duplicates(inplace=True)
print(df_disease_symptom.head())


             disease               symptom
0   fungal infection               itching
1   fungal infection             skin_rash
2   fungal infection  nodal_skin_eruptions
3   fungal infection   dischromic _patches
32           allergy   continuous_sneezing


4️⃣ Création des relations maladie → précautions (cellule code)

In [19]:
rows = []
for _, row in df_precaution.iterrows():
    disease = row['Disease']
    for i in range(1, 5):
        precaution = row[f'Precaution_{i}']
        if precaution:
            rows.append((disease, precaution))

df_disease_precaution = pd.DataFrame(rows, columns=['disease', 'precaution'])
df_disease_precaution.drop_duplicates(inplace=True)
print(df_disease_precaution.head())


         disease                precaution
0  drug reaction           stop irritation
1  drug reaction  consult nearest hospital
2  drug reaction          stop taking drug
3  drug reaction                 follow up
4        malaria  consult nearest hospital


5️⃣ Génération des fichiers CSV pour Neo4j (cellule code)

In [23]:
# Maladies
df_diseases = pd.DataFrame({'name': df_symptoms['Disease'].unique()})
df_diseases.to_csv("neo4j_import/diseases.csv", index=False)

# Symptômes
df_symptoms_unique = pd.DataFrame({'name': df_disease_symptom['symptom'].unique()})
df_symptoms_unique.to_csv("neo4j_import/symptoms.csv", index=False)

# Précautions
df_precautions_unique = pd.DataFrame({'name': df_disease_precaution['precaution'].unique()})
df_precautions_unique.to_csv("neo4j_import/precautions.csv", index=False)

# Relations
df_disease_symptom.to_csv("neo4j_import/disease_symptom.csv", index=False)
df_disease_precaution.to_csv("neo4j_import/disease_precaution.csv", index=False)
